# Simple EDA — German grid load and residual load

**Spec:** [`specs/01-Simple-EDA.md`](../specs/01-Simple-EDA.md) ·
**Data:** `data/smard.csv` (SMARD / Bundesnetzagentur, hourly, region DE)

## What this notebook is for

Understand the SMARD dataset well enough to make informed modeling decisions for the
1-day-ahead **residual load** forecast, and describe — descriptively, without defining
thresholds — where the extreme residual load cases that motivate the project actually sit.

It answers five questions:

1. Is the dataset complete, correctly typed, and continuous enough to be treated as an hourly
   time series?
2. Are our aggregation helpers correct, and what exactly do our calendar conventions mean?
3. What are the trend, seasonal and calendar structures in each series?
4. How do the series relate to each other, and which relations are candidate features?
5. What does the `residual_load` distribution look like, and how do its tails behave in time?

## How to read it

- **§6 is a contract, not a private choice.** The week convention, the season definition, the
  reporting units and the descriptive-slice policy fixed there are inherited by
  [`specs/02-Deep-EDA.md`](../specs/02-Deep-EDA.md) rather than re-derived.
- **Every plot** carries a title, axis labels and explicit units — MWh, average MW or MWh/day,
  never an unlabelled number — and is followed by one to three sentences saying what it shows.
- **No thresholds.** This notebook does not define a risk flag, a cut-off or a labelled column.
  Where it shows "the tail hours" it selects them by rank, for description only (§6.5).
- **Units.** Values are energy per hourly interval in MWh. Over an hourly interval that number
  is also the average power in MW, which is why the hourly mean of a series and its average MW
  are the same number — §6.4 makes that explicit rather than leaving it to be inferred.

---

## 1 · Setup

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it by
running [`notebooks/API-connection.ipynb`](API-connection.ipynb) top to bottom — it pulls the
SMARD API (no key required) and writes the file in German Excel CSV format
(`sep=";"`, `decimal=","`, `utf-8-sig`).

In [ ]:
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["figure.max_open_warning"] = 0  # this notebook draws ~30 figures on purpose

# Works whether the kernel starts in notebooks/ (Jupyter) or at the repo root (nbconvert).
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")
print(f"reading {DATA}")

---

## 2 · Helpers

`period_mean`, `style_timeseries` and `seasonal_plot` are **copied** from
[`notebooks/EDA-robert.ipynb`](EDA-robert.ipynb), which this spec does not modify. Three
changes apply to the copies here:

1. `ylabel` becomes a **required** argument in `style_timeseries` and `seasonal_plot`. The
   originals default it to `"(MWh)"`, which silently violates the reporting convention of §6.5.
2. `seasonal_plot` **honours** `ylabel`. The original accepts it, documents it, and then
   hard-codes `ax.set_ylabel("MWh")`.
3. `period_mean`'s edge rule moves into `_complete_periods` so it exists in exactly one place,
   shared with the new `period_energy`. The arithmetic is unchanged — §6.3 proves it.

`period_mean`'s docstring also drops the phrase *"the first and last one"*. The rule is **drop
periods the data does not fully cover**, which is not the same thing: this record starts exactly
on a month boundary, so January 2022 is complete and only the trailing month is dropped. The
code was always right; the prose was not.

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    The one edge rule of this notebook, in one place. A period counts only if it starts no
    earlier than the first observation and ends no later than the last observation's closing
    edge. Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(time_series, freq):
    """Mean of `time_series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts. Note the rule is "not fully covered", not "the first and the last":
    see the correctness test in section 6.3.

    Copied from notebooks/EDA-robert.ipynb; the edge rule was factored into `_complete_periods`
    without changing the result.
    """
    agg = time_series.groupby(time_series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(time_series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(time_series, freq, drop_incomplete=True):
    """Per-period aggregate of `time_series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view (MW). Deliberately not
        ``mwh_per_day / 24``: a month containing the spring DST switch holds 743 hours, not 744.
    ``hours``, ``days``
        the two denominators, exposed so section 6.4's comparison table needs no second copy of
        this arithmetic.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`. Pass
    ``drop_incomplete=False`` to keep them — only the 6.4 table does, because it has to *show*
    the period the rule discards.
    """
    grouped = time_series.groupby(time_series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(time_series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months. `days_in_month` would be "M"-only.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required (not defaulted to "(MWh)" as in the original): every plot must state
    whether it shows MWh, average MW or MWh/day. See the reporting convention in section 6.5.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")


def seasonal_plot(df, y_value, title, ylabel):
    """Creates a seasonal plot from a dataframe.

    Args:
        df (DataFrame): frame with separate `month` and `year` columns, already aggregated to
            one row per (year, month). Passing raw hourly data makes seaborn bootstrap a
            confidence interval per cell over 41k rows — minutes of runtime, meaningless band.
        y_value (str): name of the y-value to plot
        title (str): title of the plot
        ylabel (str): axis description, including units. Required, and actually applied — the
            original hard-coded "MWh" and ignored this argument.
    """
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=df,
        x="month",
        y=y_value,
        hue="year",
        palette="viridis",
        legend=True,
        ax=ax
    )
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.show()

---

## 3 · Load and prepare

The CSV is German Excel format, so every numeric column arrives as text with a comma decimal
separator. The failure mode to guard against is silent: unconverted columns land as a string
dtype, every aggregate still computes something, and every number is wrong. The dtype assertion
below is the guard.

The flat, `RangeIndex`ed frame is called `raw` and is **deleted** at the end of the loading
cells. Everything downstream uses `ts`, so the two cannot drift apart.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them. Note the inconsistent
# capitalisation in the source ("Grid Load" vs "Forecast Grid load") — reproduced deliberately.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid load": "fc_grid_load",
    "Forecast Residual Load": "fc_res",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

print("as read from disk — note the comma decimals and the string dtypes:")
display(raw.head(3))
display(raw.dtypes.to_frame("dtype"))

In [ ]:
raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

ts = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cells

ts.head(3)

In [ ]:
print(f"shape           : {ts.shape[0]:,} rows x {ts.shape[1]} columns")
print(f"index           : {ts.index.min()}  ->  {ts.index.max()}")
print(f"index monotonic : {ts.index.is_monotonic_increasing}, unique: {ts.index.is_unique}")
display(ts.dtypes.to_frame("dtype"))

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(pd.api.types.is_float_dtype(ts[c]) for c in COLUMNS.values()), ts.dtypes

# Snapshot for the self-check in section 11: re-asserted at the end, so a cell inserted anywhere
# in between that mutates `ts` is caught regardless of which vintage of the CSV was loaded.
LOADED = {"rows": len(ts), "start": ts.index.min(), "end": ts.index.max()}

display(ts.describe().T)

41 107 hourly rows spanning 2022-01-01 00:00 to 2026-09-09 23:00, all eight series `float64`,
nothing obviously degenerate in `describe()`. The one number worth pausing on is
`residual_load`'s minimum: it is **negative**, which is valid data (renewable oversupply), not
an error. That rules out log scales and log transforms for this series throughout.

---

## 4 · Derived columns and the `SERIES` constant

All derived columns are defined **here, in one place**, immediately after loading. Two of them
are needed by the data quality audit itself (`renewables` for the identity check, `hour` for the
night-solar check), so they cannot wait for the section that first plots them.

`SERIES` names the eight data columns. Without it, `ts.corr()` in §9 would silently pull the
derived columns into the correlation heatmap, and `describe()` would report on `year` and `dow`
as though they were measurements.

> **On "nine series".** The spec says nine; the CSV has eight numeric columns. Its Data table
> has nine rows only because it counts `timestamp`. Eight it is — `renewables` is derived, and
> appears in §9.2 where its relation to `residual_load` is the actual subject rather than a
> tautology cluttering a heatmap.

`season`/`season_year` and `spans_gap` are created here with everything else; the *reasoning*
behind them lives where it is used — the gap narrative in §5.2, the season contract in §6.5.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_res",
]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter. Rationale in 6.5.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

ts["renewables"] = ts[["wind_on", "wind_off", "solar"]].sum(axis=1)
ts["year"] = ts.index.year
ts["month"] = ts.index.month
ts["hour"] = ts.index.hour
ts["dow"] = ts.index.dayofweek
ts["is_weekend"] = ts.index.dayofweek >= 5
ts["date"] = ts.index.date
ts["season"] = pd.Categorical(
    ts.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
ts["season_year"] = ts.index.year + (ts.index.month == 12)
# True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
ts["spans_gap"] = ts.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(ts.columns) == SERIES + DERIVED, list(ts.columns)
print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {ts.shape[1]} columns")
ts[DERIVED].head(3)

`ts` now carries exactly the eight data columns plus ten declared derived ones, and the assertion
above is what keeps that true. The closing self-check in §11 re-runs it against the same two
lists — which is the mechanical proof that no flag or label column crept in along the way.

One consequence to keep in mind for the rest of the notebook: `date` is object dtype, so a bare
`ts.groupby(...).mean()` now raises under pandas 3. Every aggregation from here on names its
columns explicitly.